# Prachatai — 5 Seeds × 100 Epochs (Frozen WangchanBERTa, Kaggle T4 ×2)

- Official `prachatai_train.csv / prachatai_validation.csv / prachatai_test.csv`
- WangchanBERTa encoder **frozen**
- X-Transformers decoder + Attention Pooling + classifier
- ใช้ **Optuna best params เดิม**
- **ไม่รัน Optuna ซ้ำ**
- 5 independent runs: `[13, 21, 42, 87, 123]`
- **100 epochs ต่อ run**
- รวม repeated-run training budget = **500 decoder-training epochs**
- ใช้ official split เดิมทุก run
- Threshold tuning จาก validation หลังจบแต่ละ run
- Mean / Median / Variance / SD
- Per-category Precision / Recall / F1 ครบ 12 labels
- ถ้า Kaggle ตรวจพบ **T4 ×2** จะใช้ `torch.nn.DataParallel` อัตโนมัติ

## Cache
Prachatai มีขนาดใหญ่ จึงใช้ **disk-backed NumPy memmap**
สำหรับ hidden-state cache เพื่อลด RAM usage.
ควรเผื่อพื้นที่ดิสก์ประมาณ 14–16 GB.

> WangchanBERTa encode/cache เพียงครั้งเดียว จากนั้นทั้ง 5 runs ฝึกเฉพาะ decoder/pooling/classifier.


In [1]:
!pip install -q "transformers>=4.44,<5" sentencepiece x-transformers scikit-learn pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.3/107.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 81.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.1/143.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.7 MB/s eta 0:00:00


In [2]:
import os, glob, gc, json, random, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from x_transformers import Decoder

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, hamming_loss
)

PRIMARY_DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
GPU_COUNT = torch.cuda.device_count()
USE_DATA_PARALLEL = GPU_COUNT >= 2

print("PRIMARY_DEVICE:", PRIMARY_DEVICE)
print("GPU COUNT:", GPU_COUNT)

for i in range(GPU_COUNT):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

print("USE_DATA_PARALLEL:", USE_DATA_PARALLEL)

if GPU_COUNT < 2:
    print("WARNING: ไม่พบ GPU 2 ใบ — โค้ดจะรันด้วย device ที่มีอยู่")


PRIMARY_DEVICE: cuda:0
GPU COUNT: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
USE_DATA_PARALLEL: True


In [3]:
# =========================================================
# CONFIG
# =========================================================

MODEL_NAME = "airesearch/wangchanberta-base-att-spm-uncased"

MAX_LEN = 128
BATCH_SIZE = 64
CACHE_BATCH_SIZE = 32

EPOCHS = 100
TRAIN_SEEDS = [13, 21, 42, 87, 123]

TEXT_COLUMN = "body_text"

LABEL_COLUMNS = [
    "politics",
    "human_rights",
    "quality_of_life",
    "international",
    "social",
    "environment",
    "economics",
    "culture",
    "labor",
    "national_security",
    "ict",
    "education",
]
NUM_LABELS = len(LABEL_COLUMNS)

# Best params จาก:
# wangchanberta-x-transformers-prachatai-optimized-2(2).ipynb
FIXED_BEST_PARAMS = {
    "depth": 2,
    "heads": 4,
    "attn_dropout": 0.2691450563663464,
    "ff_dropout": 0.28475714690185233,
    "ff_mult": 4,
    "lr": 0.0001955577124628116,
    "weight_decay": 0.0015755090667191358,
}

OUTPUT_DIR = "/content" if os.path.exists("/content") else "/kaggle/working"
CACHE_DIR = os.path.join(OUTPUT_DIR, "prachatai_wangchanberta_memmap_cache")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

print("OUTPUT_DIR:", OUTPUT_DIR)
print("CACHE_DIR :", CACHE_DIR)
print("EPOCHS:", EPOCHS)
print("SEEDS:", TRAIN_SEEDS)

OUTPUT_DIR: /content
CACHE_DIR : /content/prachatai_wangchanberta_memmap_cache
EPOCHS: 100
SEEDS: [13, 21, 42, 87, 123]


In [4]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

In [5]:
# =========================================================
# FIND OFFICIAL SPLITS
# =========================================================

def find_csv(filename):
    candidates = [
        f"/content/{filename}",
        f"/mnt/data/{filename}",
    ]
    for p in candidates:
        if os.path.exists(p):
            return p

    # Colab: ค้น recursive ใน /content
    hits = glob.glob(f"/content/**/{filename}", recursive=True)
    if hits:
        return hits[0]

    # Kaggle fallback
    hits = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if hits:
        return hits[0]

    raise FileNotFoundError(f"ไม่พบ {filename}")

TRAIN_PATH = find_csv("prachatai_train.csv")
VALID_PATH = find_csv("prachatai_validation.csv")
TEST_PATH  = find_csv("prachatai_test.csv")

print(TRAIN_PATH)
print(VALID_PATH)
print(TEST_PATH)

/kaggle/input/datasets/tkorg99/prachatai/prachatai_train.csv
/kaggle/input/datasets/tkorg99/prachatai/prachatai_validation.csv
/kaggle/input/datasets/tkorg99/prachatai/prachatai_test.csv


In [6]:
# =========================================================
# LOAD DATA
# =========================================================

required = [TEXT_COLUMN] + LABEL_COLUMNS

def load_split(path):
    df = pd.read_csv(path, usecols=required, low_memory=False)
    df = df[required].copy()
    df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str).str.strip()

    for c in LABEL_COLUMNS:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

    return df.reset_index(drop=True)

train_df = load_split(TRAIN_PATH)
val_df   = load_split(VALID_PATH)
test_df  = load_split(TEST_PATH)

print("TRAIN:", len(train_df))
print("VALID:", len(val_df))
print("TEST :", len(test_df))
print("\nTRAIN LABEL COUNTS")
print(train_df[LABEL_COLUMNS].sum())

TRAIN: 54379
VALID: 6721
TEST : 6789

TRAIN LABEL COUNTS
politics             31401
human_rights         12061
quality_of_life       9037
international         6432
social                6321
environment           6157
economics             3994
culture               3279
labor                 2905
national_security     2865
ict                   2326
education             2093
dtype: int64


In [7]:
# =========================================================
# POS WEIGHT — สูตรเดิม
# =========================================================

pos = train_df[LABEL_COLUMNS].sum().values.astype(np.float32)
neg = len(train_df) - pos
pos_weights_np = neg / (pos + 1e-6)

print("POS WEIGHTS:", pos_weights_np)

POS WEIGHTS: [ 0.73176014  3.5086644   5.017373    7.4544463   7.602911    7.8320613
 12.615172   15.58402    17.719105   17.980453   22.378761   24.981367  ]


In [8]:
# =========================================================
# FROZEN ENCODER -> DISK MEMMAP CACHE
# =========================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

encoder = AutoModel.from_pretrained(MODEL_NAME).to(PRIMARY_DEVICE)
for p in encoder.parameters():
    p.requires_grad = False
encoder.eval()

HIDDEN_SIZE = encoder.config.hidden_size

class RawDataset(Dataset):
    def __init__(self, frame):
        self.texts = frame[TEXT_COLUMN].tolist()
        self.targets = frame[LABEL_COLUMNS].values.astype(np.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.targets[idx], dtype=torch.float32)
        }


def cache_paths(name):
    return {
        "hidden": os.path.join(CACHE_DIR, f"{name}_hidden_fp16.dat"),
        "mask": os.path.join(CACHE_DIR, f"{name}_mask_u8.dat"),
        "labels": os.path.join(CACHE_DIR, f"{name}_labels_fp32.dat"),
        "meta": os.path.join(CACHE_DIR, f"{name}_meta.json"),
    }


@torch.no_grad()
def precompute_to_memmap(frame, name):
    paths = cache_paths(name)
    n = len(frame)

    valid_cache = False
    if all(os.path.exists(paths[k]) for k in ["hidden","mask","labels","meta"]):
        try:
            meta = json.load(open(paths["meta"], "r", encoding="utf-8"))
            valid_cache = (
                meta["n"] == n
                and meta["max_len"] == MAX_LEN
                and meta["hidden_size"] == HIDDEN_SIZE
                and meta["num_labels"] == NUM_LABELS
            )
        except Exception:
            valid_cache = False

    if valid_cache:
        print(f"Reuse existing cache: {name}")
        return paths

    print(f"Create disk cache: {name}")

    hidden_mm = np.memmap(
        paths["hidden"], dtype=np.float16, mode="w+",
        shape=(n, MAX_LEN, HIDDEN_SIZE)
    )
    mask_mm = np.memmap(
        paths["mask"], dtype=np.uint8, mode="w+",
        shape=(n, MAX_LEN)
    )
    label_mm = np.memmap(
        paths["labels"], dtype=np.float32, mode="w+",
        shape=(n, NUM_LABELS)
    )

    loader = DataLoader(
        RawDataset(frame),
        batch_size=CACHE_BATCH_SIZE,
        shuffle=False,
        pin_memory=torch.cuda.is_available()
    )

    cursor = 0
    use_amp = PRIMARY_DEVICE.type == "cuda"

    for batch in tqdm(loader, desc=f"Encoding {name}"):
        ids = batch["input_ids"].to(PRIMARY_DEVICE, non_blocking=True)
        mask = batch["attention_mask"].to(PRIMARY_DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            h = encoder(
                input_ids=ids,
                attention_mask=mask
            ).last_hidden_state

        b = h.size(0)

        hidden_mm[cursor:cursor+b] = h.half().cpu().numpy()
        mask_mm[cursor:cursor+b] = mask.byte().cpu().numpy()
        label_mm[cursor:cursor+b] = batch["labels"].numpy()

        cursor += b

    hidden_mm.flush()
    mask_mm.flush()
    label_mm.flush()

    del hidden_mm, mask_mm, label_mm

    with open(paths["meta"], "w", encoding="utf-8") as f:
        json.dump({
            "n": n,
            "max_len": MAX_LEN,
            "hidden_size": HIDDEN_SIZE,
            "num_labels": NUM_LABELS,
        }, f, indent=2)

    return paths


train_cache = precompute_to_memmap(train_df, "train")
val_cache   = precompute_to_memmap(val_df, "validation")
test_cache  = precompute_to_memmap(test_df, "test")

del encoder
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Encoder cache complete.")

tokenizer_config.json:   0%|          | 0.00/282 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/905k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/423M [00:00<?, ?B/s]

Create disk cache: train


...

Create disk cache: validation


Encoding validation:   0%|          | 0/211 [00:00<?, ?it/s]

Create disk cache: test


Encoding test:   0%|          | 0/213 [00:00<?, ?it/s]

Encoder cache complete.


In [9]:
# =========================================================
# DISK-BACKED DATASET
# =========================================================

class MemmapDataset(Dataset):
    def __init__(self, paths, n):
        self.paths = paths
        self.n = n
        self._hidden = None
        self._mask = None
        self._labels = None

    def _open(self):
        if self._hidden is None:
            self._hidden = np.memmap(
                self.paths["hidden"], dtype=np.float16, mode="r",
                shape=(self.n, MAX_LEN, HIDDEN_SIZE)
            )
            self._mask = np.memmap(
                self.paths["mask"], dtype=np.uint8, mode="r",
                shape=(self.n, MAX_LEN)
            )
            self._labels = np.memmap(
                self.paths["labels"], dtype=np.float32, mode="r",
                shape=(self.n, NUM_LABELS)
            )

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        self._open()

        # torch.tensor ทำ copy จาก memmap เพื่อให้ writable/contiguous
        return {
            "embeddings": torch.tensor(self._hidden[idx], dtype=torch.float16),
            "attention_mask": torch.tensor(self._mask[idx], dtype=torch.bool),
            "labels": torch.tensor(self._labels[idx], dtype=torch.float32),
        }

train_dataset = MemmapDataset(train_cache, len(train_df))
val_dataset   = MemmapDataset(val_cache, len(val_df))
test_dataset  = MemmapDataset(test_cache, len(test_df))

def make_loaders(seed):
    g = torch.Generator()
    g.manual_seed(seed)

    return (
        DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            generator=g,
            num_workers=0,
            pin_memory=torch.cuda.is_available()
        ),
        DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=0,
            pin_memory=torch.cuda.is_available()
        ),
        DataLoader(
            test_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=0,
            pin_memory=torch.cuda.is_available()
        ),
    )

In [10]:
# =========================================================
# MODEL — architecture เดิม
# =========================================================

class AttentionPooling(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, x, mask):
        scores = self.attention(x).squeeze(-1)
        scores = scores.masked_fill(
            mask == 0,
            torch.finfo(scores.dtype).min
        )
        weights = torch.softmax(scores, dim=1)
        return torch.sum(x * weights.unsqueeze(-1), dim=1)


class EmotionModel(nn.Module):
    def __init__(self):
        super().__init__()
        p = FIXED_BEST_PARAMS

        self.decoder = Decoder(
            dim=HIDDEN_SIZE,
            depth=p["depth"],
            heads=p["heads"],
            attn_dropout=p["attn_dropout"],
            ff_dropout=p["ff_dropout"],
            ff_mult=p["ff_mult"]
        )

        self.pooling = AttentionPooling(HIDDEN_SIZE)
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(HIDDEN_SIZE, NUM_LABELS)

    def forward(self, embeddings, attention_mask):
        # คง behavior จาก Prachatai optimized notebook เดิม
        x = self.decoder(embeddings.float())
        x = self.pooling(x, attention_mask)
        x = self.dropout(x)
        return self.fc(x)

In [11]:
# =========================================================
# EVALUATION + THRESHOLD TUNING
# =========================================================

@torch.no_grad()
def collect_probs(model, loader):
    model.eval()
    probs_all, labels_all = [], []

    for batch in loader:
        emb = batch["embeddings"].to(PRIMARY_DEVICE, non_blocking=True)
        mask = batch["attention_mask"].to(PRIMARY_DEVICE, non_blocking=True)

        logits = model(emb, mask)
        probs = torch.sigmoid(logits)

        probs_all.append(probs.cpu().numpy())
        labels_all.append(batch["labels"].numpy())

    return np.vstack(labels_all), np.vstack(probs_all)


def optimize_thresholds(y_true, y_prob):
    best_thresholds = []

    # คง threshold grid จาก notebook เดิม
    for i in range(NUM_LABELS):
        best_thr = 0.5
        best_f1 = 0.0

        for thr in np.arange(0.1, 0.9, 0.05):
            pred = (y_prob[:, i] >= thr).astype(int)
            score = f1_score(y_true[:, i], pred, zero_division=0)

            if score > best_f1:
                best_f1 = score
                best_thr = thr

        best_thresholds.append(best_thr)

    return np.asarray(best_thresholds)


def metrics_from_probs(y_true, y_prob, thresholds):
    y_pred = (y_prob >= thresholds).astype(int)

    result = {
        "subset_accuracy": accuracy_score(y_true, y_pred),
        "micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "hamming_loss": hamming_loss(y_true, y_pred),
    }

    result["per_precision"] = precision_score(
        y_true, y_pred, average=None, zero_division=0
    )
    result["per_recall"] = recall_score(
        y_true, y_pred, average=None, zero_division=0
    )
    result["per_f1"] = f1_score(
        y_true, y_pred, average=None, zero_division=0
    )
    result["support"] = y_true.sum(axis=0).astype(int)

    return result

In [12]:
# =========================================================
# TRAIN ONE SEED — 100 epochs / final epoch model
# =========================================================

def train_one_seed(seed):
    seed_everything(seed)

    train_loader, val_loader, test_loader = make_loaders(seed)
    model = EmotionModel().to(PRIMARY_DEVICE)

    if USE_DATA_PARALLEL:
        model = nn.DataParallel(
            model,
            device_ids=list(range(GPU_COUNT))
        )
        print("DataParallel GPUs:", list(range(GPU_COUNT)))

    pos_weights = torch.tensor(
        pos_weights_np,
        dtype=torch.float32,
        device=PRIMARY_DEVICE
    )

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=FIXED_BEST_PARAMS["lr"],
        weight_decay=FIXED_BEST_PARAMS["weight_decay"]
    )

    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )

    use_amp = PRIMARY_DEVICE.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    start = time.perf_counter()

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0

        for batch in train_loader:
            emb = batch["embeddings"].to(PRIMARY_DEVICE, non_blocking=True)
            mask = batch["attention_mask"].to(PRIMARY_DEVICE, non_blocking=True)
            labels = batch["labels"].to(PRIMARY_DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = model(emb, mask)
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            total_loss += loss.item()

        # validation display at current epoch
        vy, vp = collect_probs(model, val_loader)
        vthr = optimize_thresholds(vy, vp)
        vm = metrics_from_probs(vy, vp, vthr)

        print(
            f"seed={seed} | epoch {epoch+1:03d}/{EPOCHS} | "
            f"loss={total_loss/len(train_loader):.4f} | "
            f"val_micro={vm['micro_f1']:.4f} | "
            f"val_macro={vm['macro_f1']:.4f}"
        )

    # threshold tune on validation after final epoch
    val_y, val_prob = collect_probs(model, val_loader)
    thresholds = optimize_thresholds(val_y, val_prob)

    test_y, test_prob = collect_probs(model, test_loader)
    result = metrics_from_probs(test_y, test_prob, thresholds)

    elapsed = time.perf_counter() - start

    del model, optimizer, scheduler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result, thresholds, elapsed

In [13]:
# =========================================================
# RUN ALL 5 SEEDS
# =========================================================

overall_rows = []
per_label_rows = []

for run_idx, seed in enumerate(TRAIN_SEEDS, start=1):
    print("\n" + "="*90)
    print(f"RUN {run_idx}/5 | seed={seed}")
    print("="*90)

    result, thresholds, elapsed = train_one_seed(seed)

    overall_rows.append({
        "run": run_idx,
        "seed": seed,
        "subset_accuracy": result["subset_accuracy"],
        "micro_precision": result["micro_precision"],
        "micro_recall": result["micro_recall"],
        "micro_f1": result["micro_f1"],
        "macro_precision": result["macro_precision"],
        "macro_recall": result["macro_recall"],
        "macro_f1": result["macro_f1"],
        "weighted_f1": result["weighted_f1"],
        "hamming_loss": result["hamming_loss"],
        "minutes": elapsed / 60.0,
    })

    for i, label in enumerate(LABEL_COLUMNS):
        per_label_rows.append({
            "run": run_idx,
            "seed": seed,
            "label": label,
            "precision": result["per_precision"][i],
            "recall": result["per_recall"][i],
            "f1": result["per_f1"][i],
            "support": int(result["support"][i]),
            "threshold": float(thresholds[i]),
        })

    # autosave หลังจบแต่ละ run เผื่อ Kaggle session หลุด
    pd.DataFrame(overall_rows).to_csv(
        os.path.join(OUTPUT_DIR, "prachatai_5seeds_100epochs_runs_PARTIAL.csv"),
        index=False,
        encoding="utf-8-sig"
    )
    pd.DataFrame(per_label_rows).to_csv(
        os.path.join(OUTPUT_DIR, "prachatai_5seeds_100epochs_per_category_PARTIAL.csv"),
        index=False,
        encoding="utf-8-sig"
    )

results_df = pd.DataFrame(overall_rows)
per_label_df = pd.DataFrame(per_label_rows)

display(results_df)
display(per_label_df)


RUN 1/5 | seed=13
DataParallel GPUs: [0, 1]
seed=13 | epoch 001/100 | loss=0.9193 | val_micro=0.6376 | val_macro=0.5890
seed=13 | epoch 002/100 | loss=0.6109 | val_micro=0.6878 | val_macro=0.6376
seed=13 | epoch 003/100 | loss=0.5569 | val_micro=0.6922 | val_macro=0.6406
seed=13 | epoch 004/100 | loss=0.5246 | val_micro=0.7085 | val_macro=0.6571
seed=13 | epoch 005/100 | loss=0.4978 | val_micro=0.7155 | val_macro=0.6609
seed=13 | epoch 006/100 | loss=0.4694 | val_micro=0.7212 | val_macro=0.6697
seed=13 | epoch 007/100 | loss=0.4387 | val_micro=0.7224 | val_macro=0.6679
seed=13 | epoch 008/100 | loss=0.4038 | val_micro=0.7177 | val_macro=0.6668
seed=13 | epoch 009/100 | loss=0.3749 | val_micro=0.7197 | val_macro=0.6669
seed=13 | epoch 010/100 | loss=0.3431 | val_micro=0.7188 | val_macro=0.6617
seed=13 | epoch 011/100 | loss=0.3129 | val_micro=0.7201 | val_macro=0.6640
seed=13 | epoch 012/100 | loss=0.2791 | val_micro=0.7283 | val_macro=0.6658
seed=13 | epoch 013/100 | loss=0.2471 | val

,run,seed,subset_accuracy,micro_precision,micro_recall,micro_f1,macro_precision,macro_recall,macro_f1,weighted_f1,hamming_loss,minutes
0,1,13,0.503314,0.716816,0.759590,0.737584,0.663615,0.699328,0.680706,0.737255,0.073158,71.799862
1,2,21,0.498895,0.725246,0.743267,0.734145,0.671579,0.675221,0.672510,0.731756,0.072863,71.543566
2,3,42,0.503167,0.725323,0.757686,0.741151,0.675827,0.695757,0.684949,0.741178,0.071635,71.076570
3,4,87,0.504493,0.727067,0.758320,0.742365,0.684198,0.689097,0.684928,0.741502,0.071243,71.110957
4,5,123,0.496980,0.715901,0.757776,0.736244,0.667853,0.692931,0.678922,0.735904,0.073489,71.154075


,run,seed,label,precision,recall,f1,support,threshold
0,1,13,politics,0.832018,0.877928,0.854357,3842,0.25
1,1,13,human_rights,0.653962,0.715420,0.683312,1511,0.10
2,1,13,quality_of_life,0.687662,0.707187,0.697288,1127,0.30
3,1,13,international,0.730108,0.814149,0.769841,834,0.30
4,1,13,social,0.460337,0.485425,0.472548,789,0.10
5,1,13,environment,0.774580,0.836788,0.804483,772,0.10
6,1,13,economics,0.619850,0.637765,0.628680,519,0.15
7,1,13,culture,0.546318,0.577889,0.561661,398,0.35
8,1,13,labor,0.835694,0.842857,0.839260,350,0.50
9,1,13,national_security,0.591045,0.585799,0.588410,338,0.20


In [14]:
# =========================================================
# STATISTICAL SUMMARY
# =========================================================

metric_cols = [
    "subset_accuracy",
    "micro_precision",
    "micro_recall",
    "micro_f1",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "weighted_f1",
    "hamming_loss",
]

summary_df = pd.DataFrame([
    {
        "metric": m,
        "mean": results_df[m].mean(),
        "median": results_df[m].median(),
        "variance": results_df[m].var(ddof=1),
        "std": results_df[m].std(ddof=1),
        "min": results_df[m].min(),
        "max": results_df[m].max(),
    }
    for m in metric_cols
])

per_label_summary = (
    per_label_df.groupby("label", sort=False)
    .agg(
        precision_mean=("precision","mean"),
        precision_variance=("precision", lambda x: x.var(ddof=1)),
        precision_std=("precision", lambda x: x.std(ddof=1)),
        recall_mean=("recall","mean"),
        recall_variance=("recall", lambda x: x.var(ddof=1)),
        recall_std=("recall", lambda x: x.std(ddof=1)),
        f1_mean=("f1","mean"),
        f1_median=("f1","median"),
        f1_variance=("f1", lambda x: x.var(ddof=1)),
        f1_std=("f1", lambda x: x.std(ddof=1)),
    )
    .reset_index()
)

print("5-RUN SUMMARY")
display(summary_df)

print("PER-CATEGORY SUMMARY")
display(per_label_summary)

5-RUN SUMMARY


,metric,mean,median,variance,std,min,max
0,subset_accuracy,0.501370,0.503167,1.053796e-05,0.003246,0.496980,0.504493
1,micro_precision,0.722071,0.725246,2.782339e-05,0.005275,0.715901,0.727067
2,micro_recall,0.755328,0.757776,4.603818e-05,0.006785,0.743267,0.759590
3,micro_f1,0.738298,0.737584,1.166404e-05,0.003415,0.734145,0.742365
4,macro_precision,0.672614,0.671579,6.230832e-05,0.007894,0.663615,0.684198
5,macro_recall,0.690467,0.692931,8.672032e-05,0.009312,0.675221,0.699328
6,macro_f1,0.680403,0.680706,2.643215e-05,0.005141,0.672510,0.684949
7,weighted_f1,0.737519,0.737255,1.628420e-05,0.004035,0.731756,0.741502
8,hamming_loss,0.072478,0.072863,9.669679e-07,0.000983,0.071243,0.073489


PER-CATEGORY SUMMARY


,label,precision_mean,precision_variance,precision_std,recall_mean,recall_variance,recall_std,f1_mean,f1_median,f1_variance,f1_std
0,politics,0.828189,0.000030,0.005506,0.881312,0.000006,0.002517,0.853914,0.854357,0.000008,0.002905
1,human_rights,0.661984,0.000042,0.006508,0.702184,0.000360,0.018980,0.681311,0.683312,0.000035,0.005881
2,quality_of_life,0.680439,0.000143,0.011972,0.720852,0.000465,0.021558,0.699789,0.697288,0.000046,0.006748
3,international,0.748991,0.000447,0.021134,0.802878,0.000192,0.013844,0.774724,0.775371,0.000063,0.007911
4,social,0.464834,0.000109,0.010430,0.484918,0.000783,0.027978,0.474055,0.476829,0.000085,0.009245
5,environment,0.810672,0.000465,0.021562,0.802073,0.000436,0.020875,0.805950,0.804483,0.000049,0.006993
6,economics,0.635162,0.000098,0.009909,0.627746,0.000827,0.028754,0.631032,0.631268,0.000168,0.012956
7,culture,0.555761,0.000408,0.020189,0.599497,0.000422,0.020535,0.576355,0.578089,0.000088,0.009391
8,labor,0.847249,0.000047,0.006889,0.842857,0.000029,0.005345,0.845035,0.843615,0.000025,0.004972
9,national_security,0.588816,0.001887,0.043434,0.556213,0.000600,0.024487,0.570927,0.580060,0.000396,0.019893


In [15]:
# =========================================================
# EXPORT
# =========================================================

files = {
    "runs": os.path.join(OUTPUT_DIR, "prachatai_5seeds_100epochs_runs.csv"),
    "summary": os.path.join(OUTPUT_DIR, "prachatai_5seeds_100epochs_summary.csv"),
    "per_label": os.path.join(OUTPUT_DIR, "prachatai_5seeds_100epochs_per_category.csv"),
    "per_label_summary": os.path.join(OUTPUT_DIR, "prachatai_5seeds_100epochs_per_category_summary.csv"),
}

results_df.to_csv(files["runs"], index=False, encoding="utf-8-sig")
summary_df.to_csv(files["summary"], index=False, encoding="utf-8-sig")
per_label_df.to_csv(files["per_label"], index=False, encoding="utf-8-sig")
per_label_summary.to_csv(files["per_label_summary"], index=False, encoding="utf-8-sig")

for k,v in files.items():
    print(k, "->", v)

runs -> /content/prachatai_5seeds_100epochs_runs.csv
summary -> /content/prachatai_5seeds_100epochs_summary.csv
per_label -> /content/prachatai_5seeds_100epochs_per_category.csv
per_label_summary -> /content/prachatai_5seeds_100epochs_per_category_summary.csv
